In [1]:
"""Lesson 13 companion — stub notebook for the project-local clai agent.

Open in VS Code (cells via `# %%`) or `jupyter lab`. This is intentionally
short: it introduces the agent defined in `examples/cli_agent.yaml` and
then hands off to the Makefile shortcuts for the actual REPL workflow.
"""

'Lesson 13 companion — stub notebook for the project-local clai agent.\n\nOpen in VS Code (cells via `# %%`) or `jupyter lab`. This is intentionally\nshort: it introduces the agent defined in `examples/cli_agent.yaml` and\nthen hands off to the Makefile shortcuts for the actual REPL workflow.\n'

# Lesson 13 — project-local `clai` agent (stub)

Read [`lessons/13-clai-agent-repl.md`](../lessons/13-clai-agent-repl.md)
alongside this notebook. The point of Lesson 13 is *combining* — there's
no new pydantic-ai API. We're plugging the YAML pattern from Lesson 12
into the bundled `clai` REPL.

## 1. Inspect the spec

The agent lives in `examples/cli_agent.yaml`. Read it once; the values
here drive the REPL's behavior.

In [2]:
from pathlib import Path

spec = Path("cli_agent.yaml").read_text()
print(spec)

# Project-local default agent for `clai` / `pai`.
#
# Load via:   uv run pai --agent examples/cli_agent.yaml
# Or:         make repl

model: google:gemini-3-flash-preview

instructions: |
  You are a terse assistant. Reply in well-formatted markdown.
  - Default to one short paragraph or a tight list.
  - Use web search when a question depends on recent or specific factual data
    (dates, current events, package versions, releases). Always cite sources.
  - When showing code, use fenced blocks with a language tag.

capabilities:
  - WebSearch
  - Thinking:
      effort: low



## 2. Load it the same way `clai` does

Under the hood, `pai --agent examples/cli_agent.yaml` calls
`Agent.from_file()`. You can do the exact same thing in Python:

In [3]:
from dotenv import load_dotenv

load_dotenv()

from pydantic_ai import Agent

agent = Agent.from_file("cli_agent.yaml")
agent

Agent(model=GoogleModel(), name=None, end_strategy='early', model_settings=None, output_type=<class 'str'>)

In [4]:
# Quick sanity check — confirm the agent loaded with the YAML's settings.
print("model:", agent.model)
print("output_type:", agent.output_type)

model: GoogleModel()
output_type: <class 'str'>


## 3. Use it

Three ways, ordered by ergonomics for daily use:

1. **`make repl`** — interactive REPL. This is the intended workflow.
2. **`make repl-prompt P="your question"`** — one-shot from your shell.
3. **Programmatic** — what this notebook is doing. Useful when you want
   a tested Python entry point rather than a terminal session.

In [5]:
# Programmatic one-shot (the equivalent of `make repl-prompt P="..."`).
result = await agent.run("In one line, what is pydantic-ai's `clai` for?")
print(result.output)

`clai` (pronounced "clay") is the official command-line interface for **PydanticAI**, used to chat with LLMs directly from the terminal or to spin up a web-based chat UI for testing custom agents.

### Key Capabilities
*   **Interactive Chat:** Start instant terminal sessions with various LLM providers (OpenAI, Anthropic, Gemini, etc.).
*   **Agent Web UI:** Launch a local web server (using `clai web`) to interact with your Python agents in a browser.
*   **One-shot Prompts:** Pass a prompt directly as an argument for quick answers without entering a full session.
*   **Development Tools:** Includes special commands like `/markdown` to format output and `/cp` to copy responses to the clipboard.

### Usage Example
```bash
# Start an interactive chat session
uvx clai -m openai:gpt-4o

# Launch a web UI for a local agent defined in 'my_agent.py'
clai web --agent my_agent:agent
```
*Source: [Pydantic Docs](https://pydantic.dev), [PyPI](https://pypi.org/project/clai/)*


## 4. From here

- For interactive use, exit this notebook and run `make repl`.
- For hosting beyond the terminal — `Agent.to_web()`, `Agent.to_a2a()`,
  durable execution with Temporal — see [`lessons/runtimes.md`](../lessons/runtimes.md).